In [ ]:
*SAS;
* 1. Read data directly from GitHub into SAS Viya;
filename rawdata url 'https://raw.githubusercontent.com/CiucuTheodor/SAS_Rock_Analysis/main/ClassicAltRock.csv';

proc import datafile=rawdata
    out=work.alt_rock
    dbms=csv replace;
    getnames=yes;
run;

* 2. Prepare Data for Machine Learning;
* Create a binary target variable for Logistic Regression (High Popularity > 50);
data work.ml_data;
    set work.alt_rock;
    if Popularity > 50 then High_Pop = 1; 
    else High_Pop = 0;
run;

* 3. Data Splitting: Split the data into Training (70%) and Testing (30%);
proc surveyselect data=work.ml_data out=work.ml_split 
    seed=12345 samprate=0.7 outall noprint;
run;

data work.train work.test;
    set work.ml_split;
    if selected then output work.train;
    else output work.test;
run;

* 4. Logistic Regression (SAS ML);
* Build a predictive model using stepwise selection;
title "Logistic Regression Model: Predicting High Popularity";
proc logistic data=work.train descending outmodel=work.logitmodel;
    model High_Pop = Danceability Energy Acousticness Liveness Tempo / selection=stepwise;
run;
title;

* 5. Decision Tree (SAS ML);
* Using SAS High-Performance Split to build and train a decision tree;
title "Decision Tree: Predicting Continuous Track Popularity";
proc hpsplit data=work.train seed=123;
    target Popularity;
    input Danceability Energy Acousticness Liveness Tempo / level=interval;
    prune costcomplexity;
run;
title;
